In [ ]:
%pip install catboost

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import os

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
golden = os.path.join(path, 'Q3_data.csv')
data = pd.read_csv(golden)

In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
print(data.info())

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 1: Write your code here:
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
print(missing_data.head(40))
print(data.shape)

In [ ]:
#all of those who are more than 85% missing values are useless so we will drop them
cols = ['D_87', 'D_88', 'D_39', 'D_110', 'D_111', 'D_108', 'D_42', 'D_73', 'D_135', 'D_136', 'D_138', 'D_134', 'D_137', 'R_9', 'B_29', 'D_106', 'D_106', 'D_132',
                          'D_49', 'D_66', 'D_76', 'R_26', 'D_76', 'D_142']
data = data.drop(columns=['D_87', 'D_88', 'B_39', 'D_110', 'D_111', 'D_108', 'B_42', 'D_73', 'D_135', 'D_136', 'D_138', 'D_134', 'D_137', 'R_9', 'B_29', 'D_106', 'D_106', 'D_132',
                          'D_49', 'D_66', 'D_76', 'R_26', 'D_76', 'D_142'])

In [ ]:
# Task 1: Write your code here:
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
print(missing_data.head(40))
print(data.shape)
# the data is still has so many missing values but we will fill them
fill_col = ['D_53', 'D_42', 'D_82', 'D_17', 'D_56', 'S_9', 'D_105', 'D_50', 'D_77', 'D_43', 'D_46', 'S_3', 'S_7']

In [ ]:
categorical_cols = data.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
#since they are all numbers so w will fill by the mean for any data that has more than 20% missing values

In [ ]:
for col in ['D_53', 'D_42', 'D_82', 'B_17', 'D_56', 'S_9', 'D_105', 'D_50', 'D_77', 'D_43', 'D_46', 'S_3', 'S_7', 'S_27']:
    data[col] = data[col].fillna(data[col].mean())

In [ ]:
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
print(missing_data.head(40))
print(data.shape)
#now we can dropna rows safely
data = data.dropna()
print(data.shape)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(data)
data.shape

In [ ]:
# Task 3: Write your code here:
categorical_cols = data.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

#there are no categorical cols so no need for encoding

In [ ]:
# Task 4: Write your code here:
features = data.drop(columns=['Target'])
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
pd.DataFrame(features_scaled).head(3)
print(data['Target'])

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(data, "Target")
# data is imbalance

In [ ]:
# Task 1: Write your code here:
X = features_scaled
y = data['Target']
print(X)
print(y)

In [ ]:
# Task 2,3,4,5: Write your code here:
# I will use stratified Kfold because the label is imbalance

model = CatBoostClassifier(verbose=0)


scores_accuracy = []
scores_f1 = []

# Stratified 5-Fold Cross-Validation
skf = KFold(n_splits=5)
for train_index, test_index in skf.split(X, y):
  # Split data into training and testing sets
  X_Train, X_Test = X[train_index], X[test_index]
  y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
  # Train the model
  model.fit(X_Train, y_Train)
  # Predict on the test set
  y_pred = model.predict(X_Test)

  # Calculate metrics
  scores_f1.append(f1_score(y_Test, y_pred))
  scores_accuracy.append(accuracy_score(y_Test, y_pred))
print(f"f1 score : {np.mean(scores_f1):.2f}")
print(f"accuracy score : {np.mean(scores_accuracy):.2f}")

In [ ]:
# Task 1: Write your code here:
catboost_model = model["CatBoost Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: